In [0]:
# creating some  test data 
employeesData  = [(4417411,'Aaron','Abbott','Chiropodist',date(1973,4,21),'aaron.abbottt@example.com','001-469-520-3284x942',23456,6)
]
_userSchema = """ id  integer ,
  firstName  STRING, 
  lastName   string,
  jobTitle    STRING,
  dob date, 
  email STRING , 
  phone STRING , 
  salary  integer,
  departmentId  integer 
"""
employees_Dataframe = spark.createDataFrame(employeesData,schema = _userSchema)

In [0]:

employeesTable = DeltaTable.forName(spark,'hands_on_catlog.default.employees')
employeesTable.alias("target").merge(
    employees_Dataframe.alias("source") , 
     (col('target.id') == col('source.id'))
).whenMatchedUpdate(
   set= {
       
           'firstName' : col('source.firstName') , 
           'lastName' : col('source.lastName') , 
           'jobTitle' : col('source.jobTitle') , 
           'dob'      :col('source.dob'),
           'email'     :col('source.email'), 
           'phone'    : col('source.phone') , 
           'salary'    : col('source.salary'),
           'departmentId': col('source.departmentId') , 
           'updateddate' : current_timestamp()      
   }
).whenNotMatchedInsert(
   values = {
           'firstName' : col('source.firstName') , 
           'lastName' : col('source.lastName') , 
           'jobTitle' : col('source.jobTitle') , 
           'dob'      :col('source.dob'),
           'email'     :col('source.email'), 
           'phone'    : col('source.phone') , 
           'salary'    : col('source.salary'),
           'departmentId': col('source.departmentId')
   }

).execute()

In [0]:
#  finding the matching records 
joinedDF = (
employees_Dataframe
.alias('source').
join(employeesTableSourceDF.alias('target') ,on=(col("target.id") == col('source.id')
     ), 
     how= 'left'
)
)
new_records = joinedDF.filter('target.id is null').select('source.*')
matchedupdatedRecords =( joinedDF.filter('target.id is not null').filter(xxhash64(col('source.email'))!=xxhash64(col('target.email'))).select('source.*')
)

# update the status flag for the records which have records updated value in  the source 
employeesTableObject.alias('target').merge(
    matchedupdatedRecords.alias('source') , 
    col('source.id') == col('target.id')
).whenMatchedUpdate(
    set = {
           'target.flag' : lit('Inactive')
    }
).execute()
# combining the  new records and the records which got  updated as well 
dataTobeInserted = new_records.union(matchedupdatedRecords)
dataTobeInserted.createOrReplaceTempView('recordsTobeInsertedView')
